In [ ]:
import re
import os
import nest_asyncio
import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

nest_asyncio.apply()

# =========================================================
# SETTINGS
# =========================================================
OUTPUT_FILE = "propertyguru_extracted_listings.csv"
CHECKPOINT_EVERY = 50   # fewer writes = slightly faster

# =========================================================
# COLLECT LISTING LINKS FROM SEARCH RESULT PAGES
# =========================================================
SEARCH_PAGE_START = 1
SEARCH_PAGE_END = 750
CHECKPOINT_LINKS_FILE = "propertyguru_all_listing_links.csv"

SEARCH_URL_TEMPLATE = (
    "https://www.propertyguru.com.sg/property-for-sale/{page}"
    "?__NA=true"
    "&hdbEstate=1&hdbEstate=10&hdbEstate=2&hdbEstate=20&hdbEstate=21&hdbEstate=22"
    "&hdbEstate=23&hdbEstate=24&hdbEstate=25&hdbEstate=26&hdbEstate=27&hdbEstate=28"
    "&hdbEstate=11&hdbEstate=3&hdbEstate=4&hdbEstate=5&hdbEstate=6&hdbEstate=7"
    "&hdbEstate=8&hdbEstate=9&hdbEstate=12&hdbEstate=13&hdbEstate=14&hdbEstate=15"
    "&hdbEstate=17&hdbEstate=18&hdbEstate=19"
    "&search=true&listingType=sale&isCommercial=false&locale=en"
)

async def collect_listing_links():
    all_links = set()

    # resume if checkpoint exists
    if os.path.exists(CHECKPOINT_LINKS_FILE):
        old_links_df = pd.read_csv(CHECKPOINT_LINKS_FILE).astype(object)
        if "listing_url" in old_links_df.columns:
            all_links = set(old_links_df["listing_url"].dropna().astype(str).tolist())
            print(f"Loaded existing link checkpoint with {len(all_links)} listing URLs.")

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            "pg_search_profile_fast",
            headless=False,
            viewport={"width": 1440, "height": 900},
            args=["--disable-blink-features=AutomationControlled"]
        )

        page = context.pages[0] if context.pages else await context.new_page()

        print("=" * 80)
        print("Browser opened for search pages.")
        print("If you see Cloudflare / cookie banner / 'Just a moment...', solve it manually.")
        print("Waiting 10 seconds before starting...")
        print("=" * 80)
        await page.wait_for_timeout(10000)

        for page_num in range(SEARCH_PAGE_START, SEARCH_PAGE_END + 1):
            search_url = SEARCH_URL_TEMPLATE.format(page=page_num)
            print(f"\n[Search page {page_num}/{SEARCH_PAGE_END}] Opening: {search_url}")

            try:
                await page.goto(search_url, wait_until="domcontentloaded", timeout=45000)

                title = await page.title()
                if "just a moment" in title.lower() or "attention required" in title.lower():
                    print("Cloudflare detected. Solve it manually in the browser.")
                    await page.wait_for_timeout(20000)

                await page.wait_for_timeout(2500)

                # grab all hrefs from the page
                hrefs = await page.eval_on_selector_all(
                    "a",
                    """elements => elements.map(el => el.href).filter(Boolean)"""
                )

                page_links = set()

                for href in hrefs:
                    href = href.strip()

                    # keep only actual listing detail pages
                    if re.match(r"^https://www\.propertyguru\.com\.sg/listing/[^/\s]+.*$", href):
                        page_links.add(href)

                # small interaction if nothing was found yet
                if not page_links:
                    await page.mouse.move(250, 350)
                    await page.mouse.wheel(0, 1200)
                    await page.wait_for_timeout(1500)

                    hrefs = await page.eval_on_selector_all(
                        "a",
                        """elements => elements.map(el => el.href).filter(Boolean)"""
                    )

                    for href in hrefs:
                        href = href.strip()
                        if re.match(r"^https://www\.propertyguru\.com\.sg/listing/[^/\s]+.*$", href):
                            page_links.add(href)

                before_count = len(all_links)
                all_links.update(page_links)
                added_count = len(all_links) - before_count

                print(f"Found {len(page_links)} listing links. Added {added_count} new ones.")

                if page_num % 25 == 0:
                    temp_links_df = pd.DataFrame({"listing_url": sorted(all_links)})
                    temp_links_df.to_csv(CHECKPOINT_LINKS_FILE, index=False)
                    print(f"Saved link checkpoint at page {page_num}. Total unique listing URLs: {len(temp_links_df)}")

            except Exception as e:
                print(f"Error on search page {page_num}: {e}")

        await context.close()

    final_links_df = pd.DataFrame({"listing_url": sorted(all_links)})
    final_links_df.to_csv(CHECKPOINT_LINKS_FILE, index=False)

    print("\nFinished collecting listing links.")
    print("Total unique listing URLs collected:", len(final_links_df))

    return final_links_df



# =========================================================
# HELPER FUNCTIONS
# =========================================================
def normalize_text(text):
    if not text:
        return None
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def extract_price(text):
    if not text:
        return None
    patterns = [
        r"(S\$\s?[\d,]+(?:\.\d+)?)",
        r"(S\$\s?[\d,.]+\s*(?:mil|million))"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1).strip()
    return None

def extract_bedrooms(text):
    if not text:
        return None
    patterns = [
        r"(\d+)\s+bedrooms?",
        r"(\d+)\s+beds?",
        r"(\d+)\s+bed\b"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1)
    return None

def extract_bathrooms(text):
    if not text:
        return None
    patterns = [
        r"(\d+)\s+bathrooms?",
        r"(\d+)\s+baths?",
        r"(\d+)\s+bath\b"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1)
    return None

def extract_area(text):
    if not text:
        return None
    patterns = [
        r"(\d[\d,]*)\s*(sqft|sq ft|sq\.ft|sqm|m²)"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return f"{m.group(1)} {m.group(2)}"
    return None

def extract_floor(text):
    if not text:
        return None
    patterns = [
        r"floor level[:\s]+([A-Za-z0-9 \-]+)",
        r"located on[:\s]+([A-Za-z0-9 \-]+)",
        r"(\d{1,2}(?:st|nd|rd|th)\s+floor)",
        r"(high floor|mid floor|low floor)"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1).strip()
    return None

def extract_tenure(text):
    if not text:
        return None
    patterns = [
        r"(99-Year Leasehold)",
        r"(999-Year Leasehold)",
        r"(\d{2,3}-Year Leasehold)",
        r"(Freehold)"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1).strip()
    return None

def extract_property_type(text):
    if not text:
        return None
    candidates = [
        "HDB",
        "Condo",
        "Condominium",
        "Apartment",
        "Executive Condominium",
        "Landed House",
        "Terraced House",
        "Semi-Detached House",
        "Detached House",
        "Bungalow",
        "Walk-up Apartment",
        "Cluster House"
    ]
    for c in candidates:
        if re.search(rf"\b{re.escape(c)}\b", text, re.I):
            return c
    return None

def extract_listing_id(text):
    if not text:
        return None
    patterns = [
        r"listing id[:\s]+(\d+)",
        r"property id[:\s]+(\d+)"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1)
    return None

def extract_title_from_text(text):
    if not text:
        return None
    lines = [x.strip() for x in text.split("\n") if x.strip()]
    for line in lines[:12]:
        if (
            len(line) > 5
            and "S$" not in line
            and "Property for Sale" not in line
            and "Buy" not in line
            and "Contact" not in line
        ):
            return line
    return lines[0] if lines else None

def extract_town(text):
    if not text:
        return None

    towns = [
        "Ang Mo Kio","Bedok","Bishan","Bukit Batok","Bukit Merah","Bukit Panjang",
        "Bukit Timah","Central Area","Choa Chu Kang","Clementi","Geylang","Hougang",
        "Jurong East","Jurong West","Kallang","Marine Parade","Pasir Ris","Punggol",
        "Queenstown","Sembawang","Sengkang","Serangoon","Tampines","Toa Payoh",
        "Woodlands","Yishun"
    ]

    for town in towns:
        if re.search(rf"\b{re.escape(town)}\b", text, re.I):
            return town
    return None

def extract_postal_code(text):
    if not text:
        return None

    patterns = [
        r"Singapore\s*\(?(\d{6})\)?",
        r"\b(\d{6})\b"
    ]

    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1).strip()
    return None

# =========================================================
# MAIN ENRICHMENT FUNCTION
# =========================================================
async def enrich_all_links(links_df):
    results = []

    done_urls = set()
    if os.path.exists(OUTPUT_FILE):
        old = pd.read_csv(OUTPUT_FILE).astype(object)
        if "listing_url" in old.columns:
            done_urls = set(old["listing_url"].dropna().astype(str).tolist())
            results = old.to_dict("records")
            print(f"Loaded existing checkpoint with {len(done_urls)} completed listings.")

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            "pg_detail_profile_fast",
            headless=False,
            viewport={"width": 1440, "height": 900},
            args=["--disable-blink-features=AutomationControlled"]
        )

        page = context.pages[0] if context.pages else await context.new_page()

        print("=" * 80)
        print("Browser opened.")
        print("If you see Cloudflare / cookie banner / 'Just a moment...', solve it manually.")
        print("Waiting 10 seconds before starting...")
        print("=" * 80)
        await page.wait_for_timeout(10000)

        total = len(links_df)

        for i, row in links_df.iterrows():
            url = str(row["listing_url"]).strip()

            if not url or url in done_urls:
                continue

            print(f"\n[{i+1}/{total}] Opening: {url}")

            try:
                # faster navigation
                await page.goto(url, wait_until="domcontentloaded", timeout=45000)

                # quick Cloudflare/title check
                title = await page.title()
                if "just a moment" in title.lower() or "attention required" in title.lower():
                    print("Cloudflare detected. Solve it manually in the browser.")
                    await page.wait_for_timeout(20000)
                    title = await page.title()

                # wait for real content instead of sleeping 10 seconds
                main_heading = None
                heading_selectors = ["h1", "[data-testid='listing-title']", "[class*='title']"]

                for selector in heading_selectors:
                    try:
                        await page.locator(selector).first.wait_for(timeout=4000)
                        texts = await page.locator(selector).all_inner_texts()
                        texts = [t.strip() for t in texts if t and t.strip()]
                        for h in texts:
                            if len(h) > 5 and "S$" not in h and "PropertyGuru" not in h:
                                main_heading = h
                                break
                        if main_heading:
                            break
                    except PlaywrightTimeoutError:
                        pass
                    except Exception:
                        pass

                # only do small interaction if heading wasn't found
                if not main_heading:
                    await page.mouse.move(250, 350)
                    await page.mouse.wheel(0, 900)
                    await page.wait_for_timeout(1500)

                    for selector in heading_selectors:
                        try:
                            texts = await page.locator(selector).all_inner_texts()
                            texts = [t.strip() for t in texts if t and t.strip()]
                            for h in texts:
                                if len(h) > 5 and "S$" not in h and "PropertyGuru" not in h:
                                    main_heading = h
                                    break
                            if main_heading:
                                break
                        except Exception:
                            pass

                body_text = await page.locator("body").inner_text(timeout=5000)
                body_text = normalize_text(body_text)

                if not main_heading:
                    main_heading = extract_title_from_text(body_text)

                result = {
                    "listing_url": url,
                    "page_title": title,
                    "title_extracted": main_heading,
                    "exact_address": main_heading,
                    "postal_code": extract_postal_code(body_text),
                    "price_detail": extract_price(body_text),
                    "town_detail": extract_town(body_text),
                    "bedrooms_detail": extract_bedrooms(body_text),
                    "bathrooms_detail": extract_bathrooms(body_text),
                    "area_detail": extract_area(body_text),
                    "floor_detail": extract_floor(body_text),
                    "tenure_detail": extract_tenure(body_text),
                    "property_type_detail": extract_property_type(body_text),
                    "listing_id_detail": extract_listing_id(body_text),
                    "body_text_raw": body_text[:12000],
                    "error": None
                }

                print({
                    "address": result["exact_address"],
                    "price": result["price_detail"],
                    "town": result["town_detail"],
                    "beds": result["bedrooms_detail"],
                    "baths": result["bathrooms_detail"],
                    "area": result["area_detail"],
                    "floor": result["floor_detail"]
                })

                results.append(result)
                done_urls.add(url)

            except Exception as e:
                print("Error:", e)

                results.append({
                    "listing_url": url,
                    "page_title": None,
                    "title_extracted": None,
                    "exact_address": None,
                    "postal_code": None,
                    "price_detail": None,
                    "town_detail": None,
                    "bedrooms_detail": None,
                    "bathrooms_detail": None,
                    "area_detail": None,
                    "floor_detail": None,
                    "tenure_detail": None,
                    "property_type_detail": None,
                    "listing_id_detail": None,
                    "body_text_raw": None,
                    "error": str(e)
                })
                done_urls.add(url)

            if len(done_urls) % CHECKPOINT_EVERY == 0:
                temp_df = pd.DataFrame(results).drop_duplicates(subset=["listing_url"], keep="last")
                temp_df.to_csv(OUTPUT_FILE, index=False)
                print(f"Checkpoint saved. Total completed so far: {len(temp_df)}")

        await context.close()

    final_df = pd.DataFrame(results).drop_duplicates(subset=["listing_url"], keep="last")
    final_df.to_csv(OUTPUT_FILE, index=False)
    return final_df

# =========================================================
# RUN
# =========================================================
links_df = await collect_listing_links()

print("\nNow enriching collected listing links...")
enriched_full_df = await enrich_all_links(links_df)

print("\nFinished.")
print("Final dataframe shape:", enriched_full_df.shape)

display(
    enriched_full_df[[
        "listing_url",
        "exact_address",
        "postal_code",
        "price_detail",
        "town_detail",
        "bedrooms_detail",
        "bathrooms_detail",
        "area_detail",
        "floor_detail",
        "tenure_detail",
        "property_type_detail",
        "listing_id_detail",
        "error"
    ]].head(20)
)